# Binary model evaluation

Loads the trained model and the test set, generates predictions and plots:
- Confusion matrix
- ROC curve
- Precision-Recall curve

It also evaluates the performance per rain subclass: light, moderate and heavy and violent.

Predictions are saved to `roc_proposed.npy` and `data_proposed_binary.npz`, and misclassified segments to `test_errors.csv`.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    f1_score, accuracy_score, precision_score, recall_score)
from data_pipeline import build_datasets, configure_gpu

In [ ]:
MODEL_PATH = "Binário_best_model.keras"
CSV_PATH   = "Split70-15-15.csv"
BATCH_SIZE = 32
THRESHOLD  = 0.5   

In [ ]:
# Load trained model and datasets
configure_gpu()

model = tf.keras.models.load_model(MODEL_PATH)
print(f"Modelo carregado: {MODEL_PATH}")
print(f"Total de parâmetros: {model.count_params():,}")

_, _, test_ds = build_datasets(CSV_PATH, batch_size=BATCH_SIZE)

In [ ]:
# Test set metadata (path, label, class_name, audio_id)
# Row order matches test_ds, which is not shuffled
full_df = pd.read_csv(CSV_PATH)
test_df = full_df[full_df["split"] == "test"].reset_index(drop=True)

print(f"Total de amostras no test: {len(test_df)}")
print(f"\nDistribuição por subclasse:")
print(test_df["class_name"].value_counts().to_string())

In [ ]:
# Test set predictions
print("Rodando inferência no test set...")
y_probs = model.predict(test_ds, verbose=1).flatten()
y_pred  = (y_probs >= THRESHOLD).astype(int)
y_true  = test_df["label"].values.astype(int)

# Sanity check
assert len(y_probs) == len(y_true), (
    f"Tamanhos diferentes! predict={len(y_probs)}, csv={len(y_true)}. "
    "Verifique se o test_ds está em shuffle=False."
)

# Store predictions for the per-subclass analysis
test_df["prob"] = y_probs
test_df["pred"] = y_pred

print(f"\nPredições geradas: {len(y_pred)}")
print(f"  - Preditos como no-rain (0): {(y_pred == 0).sum()}")
print(f"  - Preditos como rain (1):    {(y_pred == 1).sum()}")

In [ ]:
# Global test metrics
test_accuracy  = accuracy_score(y_true, y_pred)
test_precision = precision_score(y_true, y_pred)
test_recall    = recall_score(y_true, y_pred)
test_f1        = f1_score(y_true, y_pred)
test_auc       = auc(*roc_curve(y_true, y_probs)[:2])

print("=" * 50)
print("MÉTRICAS FINAIS NO TEST SET")
print("=" * 50)
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-score:  {test_f1:.4f}")
print(f"AUC:       {test_auc:.4f}")
print("=" * 50)
print("\nClassification report completo (sklearn):")
print(classification_report(y_true, y_pred, target_names=["no-rain", "rain"], digits=4))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=True,
            xticklabels=["no-rain (0)", "rain (1)"],
            yticklabels=["no-rain (0)", "rain (1)"],
            annot_kws={"size": 14})
plt.xlabel("Predito", fontsize=12)
plt.ylabel("Real", fontsize=12)
plt.title("Matriz de Confusão -- Test Set", fontsize=13)
plt.tight_layout()
plt.show()

print(f"True Negatives  (TN): {tn:>5}  (acertou no-rain)")
print(f"False Positives (FP): {fp:>5}  (errou: previu rain quando era no-rain)")
print(f"False Negatives (FN): {fn:>5}  (errou: previu no-rain quando era rain)")
print(f"True Positives  (TP): {tp:>5}  (acertou rain)")

In [ ]:
# ROC curve
fpr, tpr, roc_thresholds = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f"ROC (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random (AUC = 0.5)")
plt.xlabel("False Positive Rate (1 - Especificidade)")
plt.ylabel("True Positive Rate (Recall)")
plt.title("Curva ROC -- Test Set")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
np.save("roc_proposed.npy", np.column_stack([y_true.astype(np.float32), y_probs.astype(np.float32)]))
print("salvo: roc_proposed.npy  | shape:", (len(y_true), 2))

In [ ]:
# Precision-Recall curve
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_true, y_probs)
ap_score = average_precision_score(y_true, y_probs)

plt.figure(figsize=(8, 6))
plt.plot(recall_curve, precision_curve, linewidth=2,
         label=f"Precision-Recall (AP = {ap_score:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall -- Test Set")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Per-subclass performance
subclass_order = ["no-rain", "light", "moderate", "heavy", "violent"]
results = []

for subclass in subclass_order:
    sub = test_df[test_df["class_name"] == subclass]
    if len(sub) == 0:
        continue
    expected = 0 if subclass == "no-rain" else 1
    correct  = (sub["pred"] == expected).sum()
    total    = len(sub)
    results.append({
        "Subclasse":    subclass,
        "Total":        total,
        "Corretas":     correct,
        "Erradas":      total - correct,
        "Acerto (%)":   round(100 * correct / total, 2),
    })

results_df = pd.DataFrame(results)
print("Desempenho por subclasse:")
print(results_df.to_string(index=False))

In [ ]:
# Per-subclass accuracy bar chart
plt.figure(figsize=(9, 5))
bars = plt.bar(results_df["Subclasse"], results_df["Acerto (%)"],
               color=["#4c72b0", "#55a868", "#c44e52", "#8172b2", "#ccb974"])
plt.ylim([0, 105])
plt.ylabel("Acerto (%)")
plt.title("Desempenho por subclasse -- Test Set")
for bar, pct in zip(bars, results_df["Acerto (%)"]):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 1, f"{pct:.2f}%",
             ha="center", fontsize=10)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Save misclassified segments for manual inspection
errors_df = test_df[test_df["label"] != test_df["pred"]].copy()
errors_df = errors_df[["path", "label", "pred", "prob", "class_name", "audio_id"]]
errors_df.to_csv("test_errors.csv", index=False)

print(f"Total de erros no test set: {len(errors_df)} / {len(test_df)}")
print(f"Arquivo com a lista dos erros: test_errors.csv")
print(f"\nDistribuição dos erros por subclasse:")
print(errors_df["class_name"].value_counts().to_string())

In [ ]:
np.savez("data_proposed_binary.npz",
         y_true=y_true.astype(np.int8),
         y_probs=y_probs.astype(np.float32),
         audio_id=test_df["audio_id"].values.astype(str))
print("ok | segmentos:", len(y_true))